In [ ]:
# parameters
# BINDER_FAST: set N=8 for fast cloud execution
N = 20            # Hilbert space truncation (Fock dimension)
chi_MHz = 1.5     # dispersive shift chi/2pi in MHz
alpha_snap = np.pi  # phase to imprint on |3> in the demonstration


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.gates import snap_operator, apply_snap, snap_unitary_ideal
from bosonic_gates.gates.snap import snap_phase_gradient
from bosonic_gates.states import coherent_state, fock_state


## Module 4a: SNAP Gates — Selective Number-dependent Arbitrary Phase

**Learning objectives:**
- Understand how the dispersive qubit–cavity interaction makes each Fock level
  addressable at a distinct qubit frequency
- Construct the SNAP unitary $\hat{S}(\boldsymbol{\theta})$ and verify its key properties
- Visualise how SNAP imprints phases on the Wigner function of a cavity state
- Decompose a SNAP into single-level building blocks via `snap_phase_gradient`
- Understand why SNAP + displacement is a universal gate set for the cavity

**Tensor ordering:** Throughout this notebook the cavity occupies a single-mode
Fock space of dimension $N$. When the qubit is shown explicitly the tensor order
is **qubit $\otimes$ cavity** (qubit is the left factor).

---

**Sections:**
[1 Physical Origin](#sec1) ·
[2 SNAP Unitary](#sec2) ·
[3 Phase Kick Demo](#sec3) ·
[4 Universal Gate Set](#sec4) ·
[5 Building Blocks](#sec5) ·
[Exercises](#exercises)


<a id="sec1"></a>
## 1  Physical Origin of SNAP Gates

### 1.1  Dispersive Hamiltonian

When a transmon qubit (frequency $\omega_q$) is coupled to a microwave cavity
(frequency $\omega_c$) in the *dispersive regime* ($|g/\Delta| \ll 1$ with
$\Delta = \omega_q - \omega_c$), the effective Hamiltonian becomes

$$\hat{H}_{\rm disp} = \frac{\omega_q}{2}\hat{\sigma}_z
                     + \omega_c\hat{a}^\dagger\hat{a}
                     + \chi\,\hat{a}^\dagger\hat{a}\,\hat{\sigma}_z.$$

The last term — the cross-Kerr or *dispersive shift* $\chi$ — is the key.

> **Cross-reference:** See Module 1c for the derivation of $\chi = g^2/\Delta$
> from the Jaynes–Cummings model in the dispersive limit.

### 1.2  Number-dependent Qubit Frequency

The dispersive term shifts the qubit transition frequency by $n\chi$ for each
photon present in the cavity:

$$f_q^{(n)} = \frac{\omega_q + n\chi}{2\pi}.$$

Each Fock level $|n\rangle$ therefore sits at a *distinct* qubit frequency,
separated by $\chi/2\pi$.

### 1.3  Selective Qubit Drive

A microwave $\pi$-pulse tuned exactly to $f_q^{(n)}$ only inverts the qubit
when the cavity contains exactly $n$ photons.  In all other Fock sectors the
drive is off-resonance and does nothing (provided the pulse bandwidth
$\Delta f \ll \chi/2\pi$).

By applying such a selective pulse followed by a second $\pi$-pulse (to return
the qubit to its initial state), the net action on the cavity is to accumulate
a phase $\theta_n$ on $|n\rangle$ only.  Concatenating such operations for all
desired $n$ values gives the **SNAP gate**:

$$\hat{S}(\boldsymbol{\theta}) = \sum_{n=0}^{N-1} e^{i\theta_n}|n\rangle\langle n|.$$

*Reference:* Heeres *et al.*, PRL **115**, 137002 (2015);
Krastanov *et al.*, PRA **92**, 040303 (2015).

---
**Lab note.** *Typical experimental values: $\chi/2\pi \approx 0.5$–$5\,\text{MHz}$,
giving an addressable spacing of $\sim 1\,\text{MHz}$ between adjacent photon-number
peaks.  The gate time is of order $1/\chi \sim 200\,\text{ns}$–$2\,\mu\text{s}$.
Longer gates achieve better frequency selectivity but accumulate more decoherence.*


In [ ]:
# Visualise the number-dependent qubit spectrum
# Each photon number n gives a distinct qubit line at f_q + n * chi

omega_q_GHz = 6.0          # bare qubit frequency (GHz)
chi_GHz = chi_MHz * 1e-3   # dispersive shift chi/2pi (GHz)

n_vals = np.arange(min(N, 8))
f_qubit_n = omega_q_GHz + n_vals * chi_GHz  # GHz

fig, ax = plt.subplots(figsize=(8, 3.5))
for n, f in zip(n_vals, f_qubit_n):
    ax.axvline(f, lw=2, label=rf"$|{n}\rangle$")
    ax.text(f, 0.85, rf"$|{n}\rangle$", ha="center", va="bottom",
            fontsize=8, transform=ax.get_xaxis_transform())

ax.set_xlabel(r"Qubit drive frequency (GHz)")
ax.set_yticks([])
ax.set_title(rf"Number-selective qubit spectrum  ($\chi/2\pi = {chi_MHz}$ MHz)")
ax.set_xlim(omega_q_GHz - 0.2*chi_GHz, omega_q_GHz + (len(n_vals)+0.2)*chi_GHz)
plt.tight_layout()
plt.show()

print("Photon-number-selective qubit frequencies (GHz):")
for n, f in zip(n_vals, f_qubit_n):
    print(f"  n={n}: f_q^(n) = {f:.4f} GHz")


<a id="sec2"></a>
## 2  The SNAP Unitary

The SNAP gate applies independent phases to each Fock component:

$$\hat{S}(\boldsymbol{\theta}) = \sum_{n=0}^{N-1} e^{i\theta_n}|n\rangle\langle n|.$$

**Properties:**
1. **Diagonal in the Fock basis** — the matrix is diagonal with entries $e^{i\theta_n}$.
2. **Unitary** — each diagonal element has modulus 1, so $\hat{S}^\dagger\hat{S} = \mathbb{1}$.
3. **$N$ independent parameters** — one phase angle per Fock level; SNAP is the
   most general diagonal unitary on an $N$-dimensional space.


In [ ]:
# Build a SNAP that applies pi phase to |3> and pi/2 to |7>
thetas = np.zeros(N)
thetas[3] = np.pi
thetas[7] = np.pi / 2

U_snap = snap_operator(N, thetas)
print("SNAP unitary type:", type(U_snap))
print("Shape:", U_snap.shape)

# Verify unitarity: U† U = I
I_N = qt.qeye(N)
diff = (U_snap.dag() * U_snap - I_N).norm()
print(f"Unitarity check ||U†U - I|| = {diff:.2e}  (should be ~0)")


In [ ]:
# Visualise the SNAP matrix: absolute values and phases
U_mat = U_snap.full()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(np.abs(U_mat), cmap="Blues", vmin=0, vmax=1)
axes[0].set_title(r"$|U_{mn}|$ — should be identity")
axes[0].set_xlabel("Column $m$")
axes[0].set_ylabel("Row $n$")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(np.angle(np.diag(U_mat)), cmap="hsv",
                     vmin=-np.pi, vmax=np.pi, aspect="auto")
# Show only the diagonal phases as a 1-D colour bar
axes[1].set_visible(False)

ax_phase = fig.add_subplot(1, 2, 2)
ax_phase.bar(np.arange(N), np.angle(np.diag(U_mat)), color="steelblue", alpha=0.8)
ax_phase.axhline(np.pi, color="crimson", ls="--", lw=1, label=r"$\pi$")
ax_phase.axhline(np.pi/2, color="orange", ls="--", lw=1, label=r"$\pi/2$")
ax_phase.set_xlabel("Fock level $n$")
ax_phase.set_ylabel(r"Diagonal phase $\theta_n$ (rad)")
ax_phase.set_title("Phases on each Fock level")
ax_phase.legend()
ax_phase.set_ylim(-0.3, np.pi + 0.3)

plt.tight_layout()
plt.show()


<a id="sec3"></a>
## 3  Phase Kick Demonstration

We apply a SNAP gate with a single non-zero phase $\theta_3 = \alpha_{\rm snap}$
to a coherent state and observe how the imprinted phase modifies the Wigner
function.  The key signature is a change in the interference pattern: the
$|3\rangle$ Fock component picks up an extra phase, distorting the Wigner
function while leaving the Fock distribution $P(n) = |c_n|^2$ unchanged.


In [ ]:
# Initial coherent state
alpha_coh = 1.5
psi_in = coherent_state(alpha=alpha_coh, N=N).state

# SNAP: kick only |3>
thetas_kick = np.zeros(N)
thetas_kick[3] = alpha_snap   # parameter cell value

psi_out = apply_snap(psi_in, thetas_kick)

# Verify Fock distribution is unchanged
P_in  = np.abs(psi_in.full().ravel())**2
P_out = np.abs(psi_out.full().ravel())**2
print("Max change in Fock distribution:",
      np.max(np.abs(P_in - P_out)), "(should be ~0)")
print("Phase on |3> changed:",
      np.angle(psi_out.full()[3, 0]) - np.angle(psi_in.full()[3, 0]),
      f"rad  (expected {alpha_snap:.4f} rad)")


In [ ]:
# Compare Wigner functions before and after the phase kick
xvec = np.linspace(-4, 4, 100)

rho_in  = qt.ket2dm(psi_in)
rho_out = qt.ket2dm(psi_out)

W_in  = qt.wigner(rho_in,  xvec, xvec)
W_out = qt.wigner(rho_out, xvec, xvec)

vmax = max(np.max(np.abs(W_in)), np.max(np.abs(W_out)))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, W, title in zip(axes,
                         [W_in, W_out, W_out - W_in],
                         [r"Before SNAP",
                          rf"After SNAP ($\theta_3 = {alpha_snap:.2f}$)",
                          r"Difference $\Delta W$"]):
    v = vmax if title != r"Difference $\Delta W$" else np.max(np.abs(W_out - W_in))
    ax.contourf(xvec, xvec, W, levels=50, cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_title(title)
    ax.set_xlabel(r"$X = \mathrm{Re}(\alpha)$")
    ax.set_ylabel(r"$P = \mathrm{Im}(\alpha)$")
    ax.set_aspect("equal")

axes[2].set_title(r"Difference $\Delta W$ (SNAP − original)")
plt.suptitle(rf"SNAP phase kick on $|3\rangle$ of coherent state $|\alpha|={alpha_coh}$",
             fontsize=12)
plt.tight_layout()
plt.show()


**Observation.** The Fock distribution $P(n)$ is unchanged — the populations
are identical — but the Wigner function changes because the relative phases
between Fock components shift the quantum interference pattern.  This is the
essence of SNAP: it is a *phase gate* in the Fock basis.

---
**Lab note.** *In experiment you would measure the Wigner function via
heterodyne tomography (displaced parity measurements).  A clear difference
between the before and after Wigner functions — with unchanged Fock
distributions — is the smoking gun for a clean SNAP gate.*


<a id="sec4"></a>
## 4  Universal Gate Set: SNAP + Displacement

Krastanov *et al.* (PRA **92**, 040303, 2015) proved that the combination of
SNAP gates and unconditional displacements $\hat{D}(\alpha)$ is **universal**
for the cavity Hilbert space: any unitary on the $N$-dimensional Fock space can
be approximated to arbitrary precision by a finite sequence of SNAPs and
displacements.

### 4.1  Fock-basis rotation as a simple example

Consider the unitary that rotates the phase of the entire Fock ladder uniformly:

$$\hat{R}(\phi) = e^{i\phi\hat{n}} = \sum_n e^{in\phi}|n\rangle\langle n|.$$

This is *exactly* a SNAP gate with $\theta_n = n\phi$ — a linear phase ramp in
Fock number.  In the lab, $\hat{R}(\phi)$ is trivially implemented by
waiting a time $t = \phi/\omega_c$ (free evolution of the cavity).


In [ ]:
# Build the Fock-basis rotation via SNAP
phi_rot = np.pi / 4  # rotation angle
thetas_rot = np.arange(N) * phi_rot   # theta_n = n * phi

U_rot_snap = snap_operator(N, thetas_rot)

# Build the analytic form: diag(exp(i n phi))
U_rot_exact = qt.Qobj(np.diag(np.exp(1j * np.arange(N) * phi_rot)))
U_rot_exact.dims = [[N], [N]]

# Compare
diff_mat = (U_rot_snap - U_rot_exact).norm()
print(f"||U_SNAP - U_exact|| = {diff_mat:.2e}  (should be ~0)")

# Also verify as a rotation: it maps coherent state |alpha> to |alpha * exp(i phi)|
alpha_test = 2.0
psi_test = qt.coherent(N, alpha_test)
psi_rotated = U_rot_snap * psi_test

# Expected: coherent state at alpha * exp(i phi)
alpha_rotated = alpha_test * np.exp(1j * phi_rot)
psi_expected  = qt.coherent(N, alpha_rotated)
fidelity = abs(float(psi_expected.dag() * psi_rotated))**2
print(f"Fidelity of rotated coherent state: {fidelity:.6f}  (should be ~1)")


In [ ]:
# Compare matrix elements for a few Fock levels
print("Matrix element comparison (diagonal only):")
print(f"{'n':>4}  {'U_SNAP[n,n]':>20}  {'U_exact[n,n]':>20}")
for n in range(min(8, N)):
    v_snap  = U_rot_snap.full()[n, n]
    v_exact = U_rot_exact.full()[n, n]
    print(f"  {n:2d}   {v_snap.real:+.6f}{v_snap.imag:+.6f}j"
          f"   {v_exact.real:+.6f}{v_exact.imag:+.6f}j")


---
**Lab note.** *The rotation $e^{i\phi\hat{n}}$ is the simplest non-trivial
SNAP gate and is free (just wait in the rotating frame).  More useful SNAPs —
like the controlled-phase gate $e^{i\theta_k|k\rangle\langle k|}$ — are the
workhorse of bosonic qubit manipulation. SNAP + displacement can prepare any
Fock-space state and implement any cavity unitary.*


<a id="sec5"></a>
## 5  Building Blocks: `snap_phase_gradient`

The function `snap_phase_gradient(N, k, theta_k)` applies a phase $\theta_k$
to *only* the $|k\rangle$ Fock level and leaves all others unchanged:

$$\hat{S}_k(\theta_k) = \mathbb{1} + (e^{i\theta_k} - 1)|k\rangle\langle k|.$$

Any SNAP gate decomposes exactly into a product of at most $N$ such elementary
gates — one per non-zero $\theta_n$:

$$\hat{S}(\boldsymbol{\theta}) = \prod_{n=0}^{N-1} \hat{S}_n(\theta_n).$$

This product commutes because all gates are diagonal in the same basis.


In [ ]:
# Verify: product of snap_phase_gradient equals snap_operator
thetas_test = np.random.default_rng(42).uniform(0, 2*np.pi, N)

# Build the product
U_product = qt.qeye(N)
for k in range(N):
    if abs(thetas_test[k]) > 1e-14:   # skip zero phases
        U_k = snap_phase_gradient(N, k, thetas_test[k])
        U_product = U_k * U_product

# Build the direct SNAP
U_direct = snap_operator(N, thetas_test)

diff = (U_product - U_direct).norm()
print(f"||product_of_gradients - snap_operator|| = {diff:.2e}  (should be ~0)")
assert diff < 1e-10, "Decomposition failed!"
print("Decomposition verified.")


In [ ]:
# Show individual building blocks for a small example
N_small = 5
thetas_small = [0.0, np.pi/4, np.pi/2, np.pi, 3*np.pi/2]

print("Individual snap_phase_gradient building blocks:")
for k, th in enumerate(thetas_small):
    U_k = snap_phase_gradient(N_small, k, th)
    diag_k = np.angle(np.diag(U_k.full()))
    print(f"  k={k}, theta_k={th:.3f} rad → diagonal phases:",
          [f"{p:.3f}" for p in diag_k])

# Verify unitarity of each building block
print("\nUnitarity of each building block:")
for k in range(N_small):
    U_k = snap_phase_gradient(N_small, k, thetas_small[k])
    err = (U_k * U_k.dag() - qt.qeye(N_small)).norm()
    print(f"  k={k}: ||U_k U_k† - I|| = {err:.2e}")


---
**Lab note.** *In practice, applying $N$ separate qubit drives (one per Fock
level) is slow.  Modern experiments use shaped pulses that simultaneously
address multiple photon-number manifolds — effectively parallelising the product
decomposition.  The building-block picture is nevertheless useful for
understanding error budgets: errors in $\theta_k$ propagate independently.*


<a id="exercises"></a>
## Exercises

**Exercise 1.** Use `snap_operator` to build a gate that rotates the phase of
$|0\rangle$ by $\pi/3$ and leaves all other Fock levels unchanged.  Apply it to
the coherent state $|\alpha = 1.5\rangle$ and verify that the Fock distribution
is unchanged while the Wigner function differs from the original.


In [ ]:
# Exercise 1
# YOUR CODE HERE


**Exercise 2.** What `thetas` array gives the identity gate?  What `thetas`
array gives a $\pi$ rotation on *every* Fock level?  (Hint: the second case
should be a global phase and therefore physically unobservable — verify that it
leaves all expectation values unchanged.)


In [ ]:
# Exercise 2
# YOUR CODE HERE


**Exercise 2 — Solution hint (hidden in PDF output):**
Identity: `thetas = np.zeros(N)`.
Global $\pi$ phase: `thetas = np.full(N, np.pi)` → $\hat{S} = e^{i\pi}\mathbb{1}$,
unobservable.


In [ ]:
# Solution — Exercise 1
thetas_ex1 = np.zeros(N)
thetas_ex1[0] = np.pi / 3
U_ex1 = snap_operator(N, thetas_ex1)

psi_coh = coherent_state(alpha=1.5, N=N).state
psi_after = U_ex1 * psi_coh

P_before = np.abs(psi_coh.full().ravel())**2
P_after  = np.abs(psi_after.full().ravel())**2
print("Max Fock population change:", np.max(np.abs(P_before - P_after)))
print("Phase on |0> changed by:",
      np.angle(psi_after.full()[0, 0]) - np.angle(psi_coh.full()[0, 0]),
      "rad")

xvec = np.linspace(-4, 4, 80)
W_b = qt.wigner(qt.ket2dm(psi_coh),   xvec, xvec)
W_a = qt.wigner(qt.ket2dm(psi_after), xvec, xvec)
vmax = max(np.max(np.abs(W_b)), np.max(np.abs(W_a)))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, W, title in zip(axes, [W_b, W_a],
                         ["Before", rf"After SNAP $\theta_0 = \pi/3$"]):
    ax.contourf(xvec, xvec, W, levels=50, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel(r"$X$")
    ax.set_ylabel(r"$P$")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()
